# Session 7 — Your turn: Training neural networks: reading the curves

Three training runs went wrong in three different ways, and nobody labelled the curves. You will diagnose them from the numbers, and then write the rule that stops a run at the right moment.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np

rng = np.random.default_rng(5)
epochs = np.arange(1, 41)
curves = {                                                                    # (training loss, validation loss) per epoch
    "A": (0.66 - 0.002 * epochs + rng.normal(0, 0.004, 40), 0.67 - 0.002 * epochs + rng.normal(0, 0.006, 40)),
    "B": (np.exp(-epochs / 6) + 0.003, 0.30 + 0.9 * (1 - np.exp(-np.maximum(epochs - 8, 0) / 12)) + rng.normal(0, 0.01, 40)),
    "C": (2.5 + rng.normal(0, 1.2, 40).clip(-1.5, 3), 2.6 + rng.normal(0, 1.3, 40).clip(-1.5, 3.5)),
}
for name, (tr, va) in curves.items():
    print(name, "training loss: first", round(tr[0], 2), "last", round(tr[-1], 2), "| validation loss: lowest", round(va.min(), 2), "at epoch", int(va.argmin()) + 1, "last", round(va[-1], 2))
from aij_check import check, close, summary

A training loss: first 0.65 last 0.58 | validation loss: lowest 0.59 at epoch 38 last 0.59
B training loss: first 0.85 last 0.0 | validation loss: lowest 0.29 at epoch 5 last 1.14
C training loss: first 4.75 last 1.4 | validation loss: lowest 1.1 at epoch 9 last 4.49


## Task 1: Diagnose three runs

For each run choose one of `"too simple"`, `"too flexible"`, `"steps too big"` and fill the dictionary `diagnosis`. Use the numbers above (and plot the curves if you like).
Then write down, in `first_fix`, the first thing you would try for each: choose among `"more capacity"`, `"early stopping or regularisation"`, `"lower learning rate"`.

In [2]:
diagnosis = {"A": "too simple", "B": "too flexible", "C": "steps too big"}
first_fix = {"A": "more capacity", "B": "early stopping or regularisation", "C": "lower learning rate"}

In [3]:
check("A: both losses stay high and fall together: too simple", lambda: diagnosis["A"] == "too simple")
check("B: training loss goes to zero while validation loss rises: too flexible", lambda: diagnosis["B"] == "too flexible")
check("C: the loss jumps around and does not fall: steps too big", lambda: diagnosis["C"] == "steps too big")
check("each diagnosis has its matching first fix", lambda: [first_fix[k] for k in "ABC"] == ["more capacity", "early stopping or regularisation", "lower learning rate"])
summary()

PASS  A: both losses stay high and fall together: too simple
PASS  B: training loss goes to zero while validation loss rises: too flexible
PASS  C: the loss jumps around and does not fall: steps too big
PASS  each diagnosis has its matching first fix

4 of 4 checks passed.  Well done.


**Why it works.** The shape of the pair of curves is the symptom: both high and level means the model cannot fit; a widening gap means it memorises; a loss that jumps means the updates overshoot. Naming the problem first tells you which knob to turn.

## Task 2: Early stopping

Write `early_stop(val_losses, patience)` that goes through the validation losses epoch by epoch (epochs counted from 1), remembers the epoch with the lowest loss so far, and stops as soon as
`patience` epochs have passed since that best epoch. It returns `(best_epoch, stop_epoch)`; if it never stops, `stop_epoch` is the last epoch.

In [4]:
def early_stop(val_losses, patience):
    best_epoch = 1
    for epoch, loss in enumerate(val_losses, start=1):
        if loss < val_losses[best_epoch - 1]: best_epoch = epoch
        elif epoch - best_epoch >= patience: return best_epoch, epoch
    return best_epoch, len(val_losses)


example = [0.90, 0.60, 0.50, 0.45, 0.47, 0.46, 0.48, 0.50, 0.44]
print(early_stop(example, 3), early_stop(example, 5))

(4, 7) (9, 9)


In [5]:
check("patience 3 stops at epoch 7 and keeps epoch 4", lambda: early_stop(example, 3) == (4, 7))
check("patience 5 never stops and ends on the lowest loss, epoch 9", lambda: early_stop(example, 5) == (9, 9))
check("on run B it keeps an early epoch", lambda: early_stop(list(curves["B"][1]), 5)[0] < 12)
check("a larger patience never stops earlier", lambda: early_stop(list(curves["B"][1]), 2)[1] <= early_stop(list(curves["B"][1]), 8)[1] and early_stop(list(curves["B"][1]), 2)[1] > 1)
summary()

PASS  patience 3 stops at epoch 7 and keeps epoch 4
PASS  patience 5 never stops and ends on the lowest loss, epoch 9
PASS  on run B it keeps an early epoch
PASS  a larger patience never stops earlier

4 of 4 checks passed.  Well done.


**Why it works.** Patience trades a little training time for protection against noise: the loss often rises for a few epochs before it falls again (run example at epochs 5 to 8). The weights kept are those of the best epoch, not the last.